In [ ]:
import os
import pandas as pd

folder_path = "BD Quindio"
archivos = os.listdir(folder_path)

# Diccionario para guardar los dataframes
dfs = {}

for archivo in archivos:
    nombre_df = archivo.split('.')[0].replace(' ', '_')
    ruta = os.path.join(folder_path, archivo)
    if archivo.endswith('.xlsx'):
        dfs[nombre_df] = pd.read_excel(ruta)
    elif archivo.endswith('.csv'):
        dfs[nombre_df] = pd.read_csv(ruta)

# Asignar los dataframes a variables específicas si existen
Quindio_CropX = dfs.get('QuindioCropX_2612', None)
Quindio_Fijo = dfs.get('QuindioFija_1012', None)
Quindio_Manejo = dfs.get('QuindioManejo_2911', None)
Quindio_Manual = dfs.get('QuindioManual_1112', None)
Quindio_Movil = dfs.get('QuindioMovil_0412', None)

In [ ]:
Quindio_CropX_suelo= Quindio_CropX[['VWC-20cm','VWC-41cm','VWC-66cm','Temp(C)-20cm','Temp(C)-41cm','EC(dS/m)-20cm','EC(dS/m)-41cm','EC(dS/m)-66cm','Tratamiento']]
Quindio_Manejo_sf=Quindio_Manejo.drop(columns=['Fecha', 'Posición_x','Posición_y'])

In [ ]:
# Mapeo Planta -> Tratamiento 
if 'Planta' not in Quindio_Movil.columns or 'Tratamiento' not in Quindio_Movil.columns:
    raise KeyError("Quindio_Movil no contiene las columnas 'Planta' y/o 'Tratamiento'")

# Crear un DataFrame con las columnas 'Planta' y 'Tratamiento', eliminando duplicados y valores nulos
planta_trat = Quindio_Movil[['Planta', 'Tratamiento']].dropna().drop_duplicates().sort_values(['Planta','Tratamiento'])

print(planta_trat.head())

# Conteo de tratamientos por planta 
counts = planta_trat.groupby('Planta')['Tratamiento'].nunique().sort_index()

print("Número de tratamientos únicos por planta:\n")
print(counts)
print("\nListado Planta -> Tratamiento(s):\n")

# Listado de tratamientos por planta
for planta, grp in planta_trat.groupby('Planta'):
    tratamientos = grp['Tratamiento'].astype(str).unique().tolist() # Obtener tratamientos únicos
    print(f"Planta {planta}: {', '.join(tratamientos)}")
# 
planta_tratamiento_df = planta_trat.groupby('Planta')['Tratamiento'].apply(lambda x: ','.join(sorted(x.astype(str)))).reset_index()
planta_tratamiento_df.columns = ['Planta', 'Tratamientos']

In [ ]:
# Reemplazar 999 por 54 en la columna 'Planta' de Quindio_Fijo
n_before = (Quindio_Fijo['Planta'] == 999).sum()
Quindio_Fijo['Planta'].replace(999, 54, inplace=True)
n_after = (Quindio_Fijo['Planta'] == 999).sum()
print(f"Reemplazadas {n_before - n_after} ocurrencias de 999 -> 54")
print("Valores únicos (muestra) en 'Planta':", Quindio_Fijo['Planta'].unique()[:20])

In [ ]:
Quindio_Fijo['Tratamiento'] = 'T21_N2P2K0'

In [ ]:
# Ignoro los valores no convertibles 
Quindio_Manual['K_suelo_Horiba (ppm)'] = pd.to_numeric(Quindio_Manual['K_suelo_Horiba (ppm)'], errors='coerce')
Quindio_Manual['Na_suelo_Horiba (ppm)'] = pd.to_numeric(Quindio_Manual['Na_suelo_Horiba (ppm)'], errors='coerce')

Quindio_Manual['x'] = Quindio_Manual['Posicion_x'].astype('object')
Quindio_Manual['y'] = Quindio_Manual['Posicion_y'].astype('object')
Quindio_Movil['x'] = Quindio_Movil['Posicion_x'].astype('object')
Quindio_Movil['y'] = Quindio_Movil['Posicion_y'].astype('object')
Quindio_Movil['Planta']=Quindio_Movil['Planta'].astype('object')
Quindio_Movil['7in1_Conductivity[mS/cm]']=Quindio_Movil['7in1_Conductivity[uS/cm]']/1000
Quindio_Manejo['x'] = Quindio_Manejo['Posición_x'].astype('object')
Quindio_Manejo['y'] = Quindio_Manejo['Posición_y'].astype('object')
Quindio_Manual['K_suelo_Horiba (ppm)'] = Quindio_Manual['K_suelo_Horiba (ppm)'].astype('float64')
Quindio_Manual['Na_suelo_Horiba (ppm)'] = Quindio_Manual['Na_suelo_Horiba (ppm)'].astype('float64')
Quindio_Manual['Tamanno cintura (mm)'] = (Quindio_Manual['Perímetro (mm)'].astype('float64'))/3.1416
Quindio_CropX['Fecha']=Quindio_CropX['Fecha_Hora']
Quindio_Fijo['7in1_Conductivity[mS/cm]']=Quindio_Fijo['7in1_Conductivity[uS/cm]']/1000

In [ ]:
Quindio_Manual['Planta'] = Quindio_Manual['Planta'].str.extract(r'(\d+)')
Quindio_Manual.loc[Quindio_Manual['Planta'] == '1160', 'Planta'] = '160'
Quindio_Manual.loc[Quindio_Manual['Planta'] == '1161', 'Planta'] = '161'

In [ ]:
Quindio_Manual_Suelo=Quindio_Manual[['Clorofila (SPAD)', 'pH_savia',
       'K_savia (ppm)', 'Ca_savia (ppm)', 'Na_savia (ppm)', 'NO3_savia (ppm)',
       'Conductividad_savia (mS/cm)', 'pH_suelo_Horiba',
       'K_suelo_Horiba (ppm)', 'Ca_suelo_Horiba (ppm)',
       'Na_suelo_Horiba (ppm)', 'NO3_suelo_Horiba (ppm)',
       'Conductividad_suelo_Horiba (mS/cm)']]

Quindio_Manual_Suelo['K_suelo_Horiba (ppm)'] = pd.to_numeric(Quindio_Manual_Suelo['K_suelo_Horiba (ppm)'], errors='coerce')
Quindio_Manual_Suelo['Na_suelo_Horiba (ppm)'] = pd.to_numeric(Quindio_Manual_Suelo['Na_suelo_Horiba (ppm)'], errors='coerce')


Quindio_Manual_Produccion=Quindio_Manual[['Altura planta (cm)','Diametro tallo (mm)','Numero flores ','Numero frutos',
                                          'Numero frutos cosechados ','Peso frutos cosechados (Kg)','Tamanno cintura (mm)',
                                          'Tamanno altura (mm)']]

colors = ["#bfd3e6", "#9b5b4f", "#4e4151", "#dbba78", "#bb9c55", "#909195","#dc1e1e","#a02933","#716807","#717cb4"]

In [ ]:
#Quindio_Movil
Quindio_Movil['Pynamometer_Radiation[W/m2]'] = Quindio_Movil['Pynamometer_Radiation[W/m^2]']*10


Quindio_Movil_Suelo=Quindio_Movil[['7in1_Nitrogen[mg/kg]','7in1_Phosphorus[mg/kg]',
                                   '7in1_Potasium[mg/kg]','7in1_Ph[pH]','7in1_Moisture[%RH]', '7in1_S_Temperature[C]',
                                   '7in1_Conductivity[mS/cm]']]
Quindio_Movil_Ambiente=Quindio_Movil[['Pynamometer_Radiation[W/m2]',
                                      'Air_sensor_Temperature[C]', 'Air_sensor_Humidity[%RH]']]

In [ ]:
# Organizar

Quindio_Fijo_Suelo= Quindio_Fijo[['7in1_Nitrogen[mg/kg]','7in1_Phosphorus[mg/kg]',
                                   '7in1_Potasium[mg/kg]','7in1_Ph[pH]','7in1_Moisture[%RH]', '7in1_S_Temperature[°C]',
                                   '7in1_Conductivity[mS/cm]']]
Quindio_Fijo_Ambiente = Quindio_Fijo[['Pynamometer_Radiation[W/m^2]',
                                      'Air_sensor_Temperature[°C]', 'Air_sensor_Humidity[%RH]']]

### Asegurar Formato de Fecha

In [ ]:
# Asegurar formatos de fecha
Quindio_Manual['Fecha'] = pd.to_datetime(Quindio_Manual['Fecha'], dayfirst=True, errors='coerce')
Quindio_Movil['Fecha'] = pd.to_datetime(Quindio_Movil['Fecha'], dayfirst=True, errors='coerce')

Quindio_Manual['Planta'] = Quindio_Manual['Planta'].astype('Int64')
Quindio_Movil['Planta'] = Quindio_Movil['Planta'].astype('Int64')


In [ ]:
keys = ['Fecha', 'Planta', 'Tratamiento']

# Agrupar por esas columnas y combinar datos fila a fila
Quindio_Manual = Quindio_Manual.groupby(keys, as_index=False).agg(lambda x: x.ffill().bfill().iloc[0])

# Verificar duplicados eliminados
print(Quindio_Manual.shape)

C:\Users\Santiago\AppData\Local\Temp\ipykernel_13532\3316476132.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Quindio_Manual = Quindio_Manual.groupby(keys, as_index=False).agg(lambda x: x.ffill().bfill().iloc[0])


(2256, 34)


#### Verificación de Acceso por Fecha

In [ ]:
Quindio_Manual.query("Planta == 4 and Fecha == '2024-12-4'")['Peso frutos cosechados (Kg)']

C:\Users\Santiago\AppData\Local\Temp\ipykernel_13532\3167607470.py:1: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  Quindio_Manual.query("Planta == 4 and Fecha == '2024-12-4'")['Peso frutos cosechados (Kg)']


1899    0.38
Name: Peso frutos cosechados (Kg), dtype: float64

#### Eliminar Duplicados en Movil

In [ ]:
# Eliminar duplicados conservando el primero que aparezca
Quindio_Movil = Quindio_Movil.drop_duplicates(subset=['Fecha', 'Planta'], keep='first')

# Verificar resultado
print(Quindio_Movil.shape)

(1006, 26)


### Unificar BD Manual y Movil

In [ ]:
import os

# Unificar características de las bases "manual" y "movil" usando Fecha + Planta
# Resultado: df_movil_manual_unificado (DataFrame) y guardado en CSV/Excel

# Normalizar tipo Fecha (solo fecha) y Planta
Quindio_Manual['Fecha'] = pd.to_datetime(Quindio_Manual['Fecha'], errors='coerce').dt.normalize()
Quindio_Movil['Fecha'] = pd.to_datetime(Quindio_Movil['Fecha'], errors='coerce').dt.normalize()

# Asegurar que 'Planta' tiene mismo tipo (use Int64 si posible)
try:
    Quindio_Manual['Planta'] = Quindio_Manual['Planta'].astype('Int64')
except Exception:
    Quindio_Manual['Planta'] = Quindio_Manual['Planta'].astype(pd.Int64Dtype(), copy=False)

try:
    Quindio_Movil['Planta'] = Quindio_Movil['Planta'].astype('Int64')
except Exception:
    Quindio_Movil['Planta'] = Quindio_Movil['Planta'].astype(pd.Int64Dtype(), copy=False)

# Copias de trabajo (evitan modificar originales)
_manual = Quindio_Manual.copy()
_movil = Quindio_Movil.copy()

# Eliminar columnas irrelevantes/duplicadas si se desea (opcional)
# Por ejemplo, 'Time' en movil ya que usamos 'Fecha'
if 'Time' in _movil.columns:
    _movil = _movil.drop(columns=['Time'])

# Merge outer para conservar todos los registros
df_merged = pd.merge(_manual, _movil, on=['Fecha', 'Planta'], how='outer', suffixes=('_manual', '_movil'))

# Función para "coalesce" (tomar valor de movil si existe, sino manual)
cols_manual = [c for c in df_merged.columns if c.endswith('_manual')]
cols_movil = [c for c in df_merged.columns if c.endswith('_movil')]

bases_manual = {c[:-7]: c for c in cols_manual}   # quitar sufijo _manual
bases_movil  = {c[:-6]: c for c in cols_movil}    # quitar sufijo _movil

# Para cada columna presente en ambas, crear columna unificada y eliminar sufijos
for base in sorted(set(bases_manual.keys()).union(bases_movil.keys())):
    col_man = bases_manual.get(base)
    col_mov = bases_movil.get(base)
    if col_mov and col_man:
        df_merged[base] = df_merged[col_mov].combine_first(df_merged[col_man])
        df_merged.drop(columns=[col_mov, col_man], inplace=True)
    elif col_mov and not col_man:
        df_merged.rename(columns={col_mov: base}, inplace=True)
    elif col_man and not col_mov:
        df_merged.rename(columns={col_man: base}, inplace=True)

# Reordenar columnas: Fecha, Planta, Tratamiento (priorizar movil si existe), luego resto
cols = df_merged.columns.tolist()
ordered = ['Fecha', 'Planta']
# Priorizar columna 'Tratamiento' si existe (provenga de movil o manual ya unificada)
if 'Tratamiento' in cols:
    ordered.append('Tratamiento')
# Añadir el resto manteniendo el orden original
for c in cols:
    if c not in ordered:
        ordered.append(c)
df_movil_manual_unificado = df_merged[ordered]

# Guardar resultado
folder_out = "BD Quindio unificados"
os.makedirs(folder_out, exist_ok=True)
csv_path = os.path.join(folder_out, "Quindio_Movil_Manual_unificado.csv")
xlsx_path = os.path.join(folder_out, "Quindio_Movil_Manual_unificado.xlsx")
df_movil_manual_unificado.to_csv(csv_path, index=False)
df_movil_manual_unificado.to_excel(xlsx_path, index=False)

# Resultado en variable para uso posterior
df_movil_manual_unificado.head()

C:\Users\Santiago\AppData\Local\Temp\ipykernel_13532\744622987.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Quindio_Movil['Fecha'] = pd.to_datetime(Quindio_Movil['Fecha'], errors='coerce').dt.normalize()
C:\Users\Santiago\AppData\Local\Temp\ipykernel_13532\744622987.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Quindio_Movil['Planta'] = Quindio_Movil['Planta'].astype('Int64')


,Fecha,Planta,Tratamiento,Altura planta (cm),Diametro tallo (mm),Clorofila (SPAD),Numero flores,Numero frutos,Numero frutos cosechados,Peso frutos cosechados (Kg),...,Parrot_capture,Tratamiento iniciado,7in1_Conductivity[mS/cm],Pynamometer_Radiation[W/m2],Cultivo,Laboratorio,Posicion_x,Posicion_y,x,y
0,2024-07-30,1,T27_N2P2K2,21.5,NaN,34.8,NaN,NaN,NaN,NaN,...,5.0,Si,0.183,428.0,Tomatoes,Quindio,1.0,0.0,1.0,0.0
1,2024-07-30,4,T24_N2P1K1,22.0,NaN,37.5,NaN,NaN,NaN,NaN,...,5.0,Si,0.309,295.0,Tomatoes,Quindio,3.2,0.0,3.2,0.0
2,2024-07-30,7,T4_N0P0K1,18.0,NaN,37.2,NaN,NaN,NaN,NaN,...,5.0,Si,0.118,383.0,Tomatoes,Quindio,5.4,0.0,5.4,0.0
3,2024-07-30,10,T15_N1P1K1,18.5,NaN,39.5,NaN,NaN,NaN,NaN,...,5.0,Si,0.278,464.0,Tomatoes,Quindio,7.6,0.0,7.6,0.0
4,2024-07-30,15,T9_N0P2K2,22.0,NaN,40.1,NaN,NaN,NaN,NaN,...,5.0,Si,0.371,508.0,Tomatoes,Quindio,7.6,1.2,7.6,1.2


In [ ]:
import os

# Unificar características de las bases "manual" y "movil" usando Fecha + Planta
# Resultado: df_movil_manual_unificado (DataFrame) y guardado en CSV/Excel

# Normalizar tipo Fecha (solo fecha) y Planta
Quindio_Manual['Fecha'] = pd.to_datetime(Quindio_Manual['Fecha'], errors='coerce').dt.normalize()
Quindio_Movil['Fecha'] = pd.to_datetime(Quindio_Movil['Fecha'], errors='coerce').dt.normalize()

# Asegurar que 'Planta' tiene mismo tipo (use Int64 si posible)
try:
    Quindio_Manual['Planta'] = Quindio_Manual['Planta'].astype('Int64')
except Exception:
    Quindio_Manual['Planta'] = Quindio_Manual['Planta'].astype(pd.Int64Dtype(), copy=False)

try:
    Quindio_Movil['Planta'] = Quindio_Movil['Planta'].astype('Int64')
except Exception:
    Quindio_Movil['Planta'] = Quindio_Movil['Planta'].astype(pd.Int64Dtype(), copy=False)

# Copias de trabajo (evitan modificar originales)
_manual = Quindio_Manual.copy()
_movil = Quindio_Movil.copy()

# Eliminar columnas irrelevantes/duplicadas si se desea (opcional)
# Por ejemplo, 'Time' en movil ya que usamos 'Fecha'
if 'Time' in _movil.columns:
    _movil = _movil.drop(columns=['Time'])

# Merge outer para conservar todos los registros
df_merged = pd.merge(_manual, _movil, on=['Fecha', 'Planta'], how='outer', suffixes=('_manual', '_movil'))

# Función para "coalesce" (tomar valor de movil si existe, sino manual)
cols_manual = [c for c in df_merged.columns if c.endswith('_manual')]
cols_movil = [c for c in df_merged.columns if c.endswith('_movil')]

bases_manual = {c[:-7]: c for c in cols_manual}   # quitar sufijo _manual
bases_movil  = {c[:-6]: c for c in cols_movil}    # quitar sufijo _movil

# Para cada columna presente en ambas, crear columna unificada y eliminar sufijos
for base in sorted(set(bases_manual.keys()).union(bases_movil.keys())):
    col_man = bases_manual.get(base)
    col_mov = bases_movil.get(base)
    if col_mov and col_man:
        df_merged[base] = df_merged[col_mov].combine_first(df_merged[col_man])
        df_merged.drop(columns=[col_mov, col_man], inplace=True)
    elif col_mov and not col_man:
        df_merged.rename(columns={col_mov: base}, inplace=True)
    elif col_man and not col_mov:
        df_merged.rename(columns={col_man: base}, inplace=True)

# Reordenar columnas: Fecha, Planta, Tratamiento (priorizar movil si existe), luego resto
cols = df_merged.columns.tolist()
ordered = ['Fecha', 'Planta']
# Priorizar columna 'Tratamiento' si existe (provenga de movil o manual ya unificada)
if 'Tratamiento' in cols:
    ordered.append('Tratamiento')
# Añadir el resto manteniendo el orden original
for c in cols:
    if c not in ordered:
        ordered.append(c)
df_movil_manual_unificado = df_merged[ordered]

# Guardar resultado
folder_out = "BD Quindio unificados"
os.makedirs(folder_out, exist_ok=True)
csv_path = os.path.join(folder_out, "Quindio_Movil_Manual_unificado.csv")
xlsx_path = os.path.join(folder_out, "Quindio_Movil_Manual_unificado.xlsx")
df_movil_manual_unificado.to_csv(csv_path, index=False)
df_movil_manual_unificado.to_excel(xlsx_path, index=False)

# Resultado en variable para uso posterior
df_movil_manual_unificado.head()

C:\Users\Santiago\AppData\Local\Temp\ipykernel_13532\744622987.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Quindio_Movil['Fecha'] = pd.to_datetime(Quindio_Movil['Fecha'], errors='coerce').dt.normalize()
C:\Users\Santiago\AppData\Local\Temp\ipykernel_13532\744622987.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Quindio_Movil['Planta'] = Quindio_Movil['Planta'].astype('Int64')


,Fecha,Planta,Tratamiento,Altura planta (cm),Diametro tallo (mm),Clorofila (SPAD),Numero flores,Numero frutos,Numero frutos cosechados,Peso frutos cosechados (Kg),...,Parrot_capture,Tratamiento iniciado,7in1_Conductivity[mS/cm],Pynamometer_Radiation[W/m2],Cultivo,Laboratorio,Posicion_x,Posicion_y,x,y
0,2024-07-30,1,T27_N2P2K2,21.5,NaN,34.8,NaN,NaN,NaN,NaN,...,5.0,Si,0.183,428.0,Tomatoes,Quindio,1.0,0.0,1.0,0.0
1,2024-07-30,4,T24_N2P1K1,22.0,NaN,37.5,NaN,NaN,NaN,NaN,...,5.0,Si,0.309,295.0,Tomatoes,Quindio,3.2,0.0,3.2,0.0
2,2024-07-30,7,T4_N0P0K1,18.0,NaN,37.2,NaN,NaN,NaN,NaN,...,5.0,Si,0.118,383.0,Tomatoes,Quindio,5.4,0.0,5.4,0.0
3,2024-07-30,10,T15_N1P1K1,18.5,NaN,39.5,NaN,NaN,NaN,NaN,...,5.0,Si,0.278,464.0,Tomatoes,Quindio,7.6,0.0,7.6,0.0
4,2024-07-30,15,T9_N0P2K2,22.0,NaN,40.1,NaN,NaN,NaN,NaN,...,5.0,Si,0.371,508.0,Tomatoes,Quindio,7.6,1.2,7.6,1.2


In [ ]:
df_movil_manual_unificado.shape

(2476, 50)

In [ ]:
print(df_movil_manual_unificado.dropna())

Empty DataFrame
Columns: [Fecha, Planta, Tratamiento, Altura planta (cm), Diametro tallo (mm), Clorofila (SPAD), Numero flores , Numero frutos, Numero frutos cosechados , Peso frutos cosechados (Kg), Perímetro (mm), Tamanno altura (mm), Frutos de 2da, pH_savia, K_savia (ppm), Ca_savia (ppm), Na_savia (ppm), NO3_savia (ppm), Conductividad_savia (mS/cm), Fosforo_savia (ppm), pH_suelo_Horiba, K_suelo_Horiba (ppm), Ca_suelo_Horiba (ppm), Na_suelo_Horiba (ppm), NO3_suelo_Horiba (ppm), Conductividad_suelo_Horiba (mS/cm), Fosforo_suelo_Hanna (ppm), Tamanno cintura (mm), 7in1_Nitrogen[mg/kg], 7in1_Phosphorus[mg/kg], 7in1_Potasium[mg/kg], 7in1_Ph[pH], 7in1_Moisture[%RH], 7in1_S_Temperature[C], 7in1_Conductivity[uS/cm], Pynamometer_Radiation[W/m^2], Air_sensor_Temperature[C], Air_sensor_Humidity[%RH], RGB_capture, Tapo_capture, Parrot_capture, Tratamiento iniciado, 7in1_Conductivity[mS/cm], Pynamometer_Radiation[W/m2], Cultivo, Laboratorio, Posicion_x, Posicion_y, x, y]
Index: []

[0 rows x 50 c

In [ ]:
df_movil_manual_unificado['Tratamiento'].describe()

count           2476
unique            30
top       T24_N2P1K1
freq              85
Name: Tratamiento, dtype: object

In [ ]:
columnas_vacias = df_movil_manual_unificado.columns[df_movil_manual_unificado.isnull().all()]
columnas_vacias

Index(['Diametro tallo (mm)'], dtype='object')

In [ ]:
df_movil_manual_unificado.describe()

,Fecha,Planta,Altura planta (cm),Diametro tallo (mm),Clorofila (SPAD),Numero flores,Numero frutos,Numero frutos cosechados,Peso frutos cosechados (Kg),Perímetro (mm),...,Pynamometer_Radiation[W/m^2],Air_sensor_Temperature[C],Air_sensor_Humidity[%RH],RGB_capture,Tapo_capture,Parrot_capture,7in1_Conductivity[mS/cm],Pynamometer_Radiation[W/m2],Posicion_x,Posicion_y
count,2476,2476.0,336.000000,0.0,592.000000,473.000000,502.000000,1975.000000,1975.000000,1795.000000,...,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,1006.000000,2476.000000,2476.000000
mean,2024-10-22 19:56:53.893376512,89.315428,115.802083,NaN,53.044426,16.173362,5.229084,2.899241,0.429088,170.762961,...,37.167594,26.730517,65.040656,0.989066,0.988072,4.966203,0.221776,371.675944,4.797940,8.281745
min,2024-07-30 00:00:00,1.0,13.000000,NaN,25.800000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.100000,20.400000,39.900000,0.000000,0.000000,0.000000,0.000000,41.000000,1.000000,0.000000
25%,2024-10-02 00:00:00,44.0,42.000000,NaN,48.975000,10.000000,2.000000,1.000000,0.143000,180.000000,...,18.350000,24.700000,57.400000,1.000000,1.000000,5.000000,0.067000,183.500000,3.200000,3.600000
50%,2024-10-30 00:00:00,88.0,117.500000,NaN,54.500000,16.000000,5.000000,3.000000,0.366000,205.000000,...,28.000000,26.400000,65.000000,1.000000,1.000000,5.000000,0.152000,280.000000,4.800000,8.400000
75%,2024-11-20 00:00:00,134.0,182.000000,NaN,57.925000,22.000000,8.000000,4.000000,0.633000,220.000000,...,54.300000,28.600000,72.500000,1.000000,1.000000,5.000000,0.283750,543.000000,7.600000,13.200000
max,2024-12-11 00:00:00,180.0,260.000000,NaN,70.000000,38.000000,24.000000,22.000000,2.486000,355.000000,...,99.100000,32.800000,98.700000,1.000000,1.000000,5.000000,6.441000,991.000000,8.600000,16.800000
std,NaN,51.93141,71.373259,NaN,7.399328,8.409763,4.274666,2.471617,0.370102,85.834704,...,22.417972,2.515012,10.085782,0.104046,0.108618,0.338053,0.307678,224.179717,2.493907,5.184029
